# Objetivo 3 — CNN Clasica: ResNet-18 (Linea Base)
Fine-tuning completo desde ImageNet.  
Capa fc adaptada: `512 → 4 → n_classes`  
El cuello de botella de **4 features** iguala la dimension de salida de los 4 qubits de los modelos HQCNN, haciendo la comparacion justa.
**Datasets:** etl_output/chest_xray · etl_output/lung_cancer

## 0 · Instalacion de dependencias

In [4]:
import subprocess, sys
pkgs = [
    'torch torchvision --index-url https://download.pytorch.org/whl/cu118',
    'scikit-learn matplotlib seaborn tqdm',
]
for pkg in pkgs:
    subprocess.run([sys.executable,'-m','pip','install','-q']+pkg.split(), check=False)
print('OK')


OK


## 1 · Imports y seed global

In [5]:
import os, random, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm.notebook import tqdm
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device: ', DEVICE)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


PyTorch: 2.12.0+cpu
Device:  cpu


## 2 · Rutas y DataLoaders

In [6]:
BASE_DIR  = Path(os.getcwd())
CHEST_OUT = BASE_DIR / 'etl_output' / 'chest_xray'
LUNG_OUT  = BASE_DIR / 'etl_output' / 'lung_cancer'
BATCH_SIZE = 32
IMG_SIZE   = (128, 128)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = T.Compose([
    T.Resize(IMG_SIZE), T.Grayscale(num_output_channels=3),
    T.RandomRotation(10), T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.15),
    T.RandomAffine(degrees=0, scale=(0.90, 1.10)),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
transform_eval = T.Compose([
    T.Resize(IMG_SIZE), T.Grayscale(num_output_channels=3),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def make_loaders(root, weighted=False):
    root = Path(root)
    ds_tr = ImageFolder(root/'train', transform=transform_train)
    ds_va = ImageFolder(root/'val',   transform=transform_eval)
    ds_te = ImageFolder(root/'test',  transform=transform_eval)
    if weighted:
        tgts = torch.tensor(ds_tr.targets)
        sw   = (1.0/torch.bincount(tgts).float())[tgts]
        loader_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE,
                               sampler=WeightedRandomSampler(sw,len(sw),True),
                               num_workers=2, pin_memory=True)
    else:
        loader_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=2, pin_memory=True)
    loader_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    loader_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return loader_tr, loader_va, loader_te, ds_tr.class_to_idx

chest_train_loader, chest_val_loader, chest_test_loader, chest_classes = make_loaders(CHEST_OUT)
lung_train_loader,  lung_val_loader,  lung_test_loader,  lung_classes  = make_loaders(LUNG_OUT, weighted=True)
N_CLASSES_CHEST = len(chest_classes)
N_CLASSES_LUNG  = len(lung_classes)

ds_lung_tr = ImageFolder(LUNG_OUT/'train')
tgts = torch.tensor(ds_lung_tr.targets)
lung_cw = (1.0/torch.bincount(tgts).float())
lung_cw = (lung_cw/lung_cw.sum()).to(DEVICE)

print('Chest:', chest_classes)
print('Lung: ', lung_classes)
print('Lung class weights:', lung_cw)


Chest: {'NORMAL': 0, 'PNEUMONIA': 1}
Lung:  {'Benign': 0, 'Malignant': 1, 'Normal': 2}
Lung class weights: tensor([0.6654, 0.1426, 0.1921])


## 3 · Arquitectura ResNet-18 adaptada
Se reemplaza la capa `fc` original (`512 → 1000`) por:
```
Linear(512, 4)  →  ReLU  →  Linear(4, n_classes)
```
El cuello de **4 features** iguala la dimension de salida de los 4 qubits (4 valores `<Z>`) de los modelos HQCNN.

In [7]:
N_QUANTUM_DIM = 4   # mismo que N_QUBITS en los modelos HQCNN

def build_resnet18(n_classes: int) -> nn.Module:
    """
    ResNet-18 preentrenado en ImageNet, fine-tuning completo.
    fc: 512 -> 4 -> n_classes
    El bottleneck de 4 features iguala la dimension de salida
    de los 4 qubits de los modelos HQCNN (comparacion justa).
    """
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features  # 512
    model.fc = nn.Sequential(
        nn.Linear(in_features, N_QUANTUM_DIM),
        nn.ReLU(),
        nn.Linear(N_QUANTUM_DIM, n_classes),
    )
    return model

# Verificacion de shapes
with torch.no_grad():
    m2 = build_resnet18(2)
    m3 = build_resnet18(3)
    x  = torch.randn(2, 3, 128, 128)
    print('ResNet-18 output (n=2):', m2(x).shape)
    print('ResNet-18 output (n=3):', m3(x).shape)
    p2 = sum(p.numel() for p in m2.parameters() if p.requires_grad)
    print(f'Parametros entrenables: {p2:,}')
    print()
    print('Capa fc:')
    print(m2.fc)


ResNet-18 output (n=2): torch.Size([2, 2])
ResNet-18 output (n=3): torch.Size([2, 3])
Parametros entrenables: 11,178,574

Capa fc:
Sequential(
  (0): Linear(in_features=512, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
)


## 4 · Diagrama de la capa fc

In [8]:
print('Arquitectura fc (ResNet-18 adaptada):')
print()
print('  avgpool  →  flatten(512)  →  Linear(512,4)  →  ReLU  →  Linear(4, n_classes)')
print()
print('       ↑ backbone ResNet-18 (congelado/fino)      ↑ comparable a 4 expval <Z> HQCNN')
print()
print('Modelos HQCNN:  backbone → pre_quantum(N_QUBITS=4) → q_layer(4 expval) → Linear(4, n_classes)')
print('ResNet-18:      backbone →                           fc(512→4)          → Linear(4, n_classes)')


Arquitectura fc (ResNet-18 adaptada):

  avgpool  →  flatten(512)  →  Linear(512,4)  →  ReLU  →  Linear(4, n_classes)

       ↑ backbone ResNet-18 (congelado/fino)      ↑ comparable a 4 expval <Z> HQCNN

Modelos HQCNN:  backbone → pre_quantum(N_QUBITS=4) → q_layer(4 expval) → Linear(4, n_classes)
ResNet-18:      backbone →                           fc(512→4)          → Linear(4, n_classes)
